# Langfuse · The Cookbook Assistant in Production

`07_langsmith` evaluated `05_llamaindex`'s cookbook assistant and fixed it in four places. This project runs the fixed assistant through **Langfuse**, and shows only what 07 did not: the parts of running an agent that matter once it is in use, not under test.

1. **Every LlamaIndex step, traced through OpenTelemetry.** Langfuse is an OpenTelemetry tracer, so LlamaIndex's own instrumentation shows each query, retrieval, embedding and model call nested where it ran. That is the route that lost every step inside a LangSmith experiment in 07.
2. **A cost on every model call.** Langfuse prices each call from a table of model prices. gpt-oss-120b on Groq is not in its table, so Groq's prices go in, and every trace shows dollars, not only tokens.
3. **The agent's prompt as versions.** Its instructions live in Langfuse as prompt `cookbook-agent`: version 1 is 05's, version 2 a revision made after 07's round 3. Every model call records the version it ran with.
4. **Monitoring through the metrics API:** cost, tokens and time, grouped by prompt version, read into this notebook.

The work: the three agent questions from 07's dataset, run once with each prompt version. That is small on purpose. One agent question takes 10,000–45,000 Groq tokens, and the free tier allows 200,000 a day, so three questions twice is about a day. Comparing prompt versions properly needs many more questions first, and, because an agent's runs vary, a few runs of each to see how consistently each version succeeds. This shows how it is done; each result is one sample. The answers are scored the way 07 scores them, with the same judge's "points made", which was checked against hand marks there.

| File | What it is |
|---|---|
| `cookbook_ops.py` | the tracing, the price, the prompt versions, the runs, the scores, the metrics queries |
| `test_ops.py` | offline tests, against a scripted Groq and a fake Langfuse client |

---
# Part 1 · Build

## 1 · Tracing: OpenTelemetry, and nothing else

Langfuse 4's client installs an OpenTelemetry tracer provider. LlamaIndex's instrumentation (`openinference`, added to the environment in 07) writes its spans to whatever provider is installed. That is the whole setup: no wrapper around the model client, and no decorator on the tools, as 07 needed for LangSmith.

```python
# cookbook_ops.py
def setup():
    """Langfuse's client, with LlamaIndex's OpenTelemetry instrumentation sending to it. Once per process.

    Langfuse 4 is an OpenTelemetry tracer: its client installs the tracer provider, and LlamaIndex's instrumentation
    (openinference) writes its spans to whatever provider is installed, so every LlamaIndex step lands in Langfuse
    -- query, retrieval, embedding, each model call -- nested under the span that was open when it ran.
    """
    global _SET_UP
    from langfuse import get_client
    client = get_client()
    if not _SET_UP:
        from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
        LlamaIndexInstrumentor().instrument()
        _SET_UP = True
    return client
```

## 2 · Cost: the model's price

Langfuse computes a call's cost from its tokens and the price of the model it names. Its table has OpenAI's prices for OpenAI's models, and nothing for gpt-oss-120b served by Groq, so a call to it shows tokens and no cost. One entry fixes that, matched by the model name LlamaIndex reports:

```python
# cookbook_ops.py
# Groq's prices for openai/gpt-oss-120b, from its model page (console.groq.com/docs/model/openai/gpt-oss-120b) on
# 8 October 2026: $0.15 per million input tokens, $0.60 per million output. Langfuse wants USD per unit.
GROQ_PRICE = {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000}
```

```python
# cookbook_ops.py
def ensure_price(client):
    """The model's price in the project's model table, added if missing. Returns the entry.

    The table holds hundreds of Langfuse's own entries, so it is read page by page: on the first page alone, this
    entry is missed and creating it again is refused ("already exists in project")."""
    page = 1
    while True:
        listed = client.api.models.list(page=page, limit=100)
        found = [m for m in listed.data if m.model_name == A.MODEL]
        if found:
            return found[0]
        if page >= listed.meta.total_pages:
            break
        page += 1
    return client.api.models.create(model_name=A.MODEL, match_pattern=r"(?i)^(openai/)?gpt-oss-120b$", unit="TOKENS",
                                    input_price=GROQ_PRICE["input"], output_price=GROQ_PRICE["output"])
```

## 3 · The prompt, as versions

The agent's instructions are taken out of the code and kept in Langfuse, where they can change without a deploy, and every change is a numbered version. Version 2 was written from round 3's misses in 07. That makes it a fair test of whether the revision does what it was written for, but not of whether it helps on questions it was not written from:

```python
# cookbook_ops.py
# Version 2, written after 07's round 3. Its LangGraph answer was right but left out what is true of BOTH designs
# (durability stops at the workflow) and what the case file gives up, and called a Denied the case file's "only"
# failure mode. The additions are general -- nothing names LangGraph -- but they were written from misses on these
# very questions, so a better score on them shows the revision does what it was written for, not that it
# generalises. That needs questions it was not written from.
PROMPTS = {
    1: A.AGENT_PROMPT,
    2: A.AGENT_PROMPT + """

When the question compares two things, cover each side in turn: how it works, what can go wrong with it, and what it
gives up. Say plainly when something holds for both. Do not call anything the only one of its kind unless a source
says so.""",
}
```

Each run asks Langfuse for the version, gives its text to 05's agent (`agent(system_prompt=...)`), and runs inside `propagate_attributes(prompt=...)`. That puts the version on every step made inside it, including the model calls LlamaIndex's instrumentation makes. So Langfuse links each call to the version that wrote its instructions, with no code of ours on the spans. Each answer is saved locally before anything else happens to it, as in 07.

```python
# cookbook_ops.py
def ensure_prompts(client):
    """Prompt `cookbook-agent`, versions 1 and 2 as in PROMPTS, created in order if missing. Langfuse numbers
    versions itself, so each is checked to have the text it should."""
    have = {}
    for version in sorted(PROMPTS):
        try:
            have[version] = client.get_prompt(PROMPT, version=version, cache_ttl_seconds=0, max_retries=0)
        except Exception:                                   # noqa: BLE001 -- not created yet
            have[version] = client.create_prompt(name=PROMPT, prompt=PROMPTS[version], labels=[f"v{version}"],
                                                 commit_message="05's AGENT_PROMPT" if version == 1 else
                                                 "after 07's round 3: both sides, what each gives up")
        assert have[version].version == version and have[version].prompt == PROMPTS[version], version
    return have

async def run_version(client, index, version, chat_model=A.chat_model, environment=None, log=print):
    """The agent questions not yet answered with this prompt version, one at a time, each its own trace.

    `propagate_attributes` puts the prompt version, the tag and the trace name on every observation made inside it,
    LlamaIndex's included: that is how each model call is linked to the version that wrote its instructions.
    `environment` keeps stand-in runs apart from real ones in every view and query.
    """
    from langfuse import propagate_attributes
    prompt = client.get_prompt(PROMPT, version=version)
    for q in AGENT_QUESTIONS:
        if q["question"] in saved(version):
            continue
        llm = chat_model(budget=A.Budget(BUDGET, f"08 v{version}"))
        began = time.monotonic()
        with propagate_attributes(prompt=prompt, tags=[f"prompt v{version}"], trace_name="cookbook-agent",
                                  environment=environment), \
                client.start_as_current_observation(as_type="span", name="cookbook-agent",
                                                    input={"question": q["question"]}) as span:
            run = await A.run_agent(A.agent(index, llm, system_prompt=prompt.prompt), q["question"])
            span.update(output={"answer": run["answer"]}, metadata={"stopped": run["stopped"]})
            trace_id = client.get_current_trace_id()
        if run["stopped"] and "tokens per day" in run["stopped"]:
            raise GroqDayLimit(run["stopped"])
        _save(version, q["question"], {"kind": "agent", "answer": run["answer"], "stopped": run["stopped"],
                                       "calls": run["calls"], "usage": llm.budget.summary(), "trace": trace_id,
                                       "seconds": round(time.monotonic() - began, 1)})
        log(f"    v{version}: {q['question'][:60]} -- {llm.budget.spent:,} tokens")
    client.flush()
```

## 4 · Monitoring: the metrics API

One query shape for every question asked of the traces: what to measure (`totalCost`, `totalTokens`, `latency`...), how to aggregate it, what to group by (`promptVersion`, `name`...), and filters. The dashboards in Langfuse's web app are built on the same queries; here they come back as rows.

```python
# cookbook_ops.py
def metrics(client, dimensions, measures, filters=(), view="observations", since=timedelta(days=7),
            environment="default"):
    """One metrics-API query: `measures` as (measure, aggregation) pairs, grouped by `dimensions`, in one
    environment (stand-in runs are kept out). Rows as dicts."""
    now = datetime.now(timezone.utc)
    query = {"view": view, "dimensions": [{"field": d} for d in dimensions],
             "metrics": [{"measure": m, "aggregation": a} for m, a in measures],
             "filters": [*filters, {"column": "environment", "operator": "=", "value": environment, "type": "string"}],
             "fromTimestamp": (now - since).isoformat(), "toTimestamp": now.isoformat()}
    return client.api.metrics.metrics(query=json.dumps(query)).data

def tagged(version):
    """A filter: only what ran with this prompt version."""
    return {"column": "tags", "operator": "any of", "value": [f"prompt v{version}"], "type": "arrayOptions"}
```

Scores are 07's: the judge's "points made", put on each answer's trace, where Langfuse's metrics can group them too.

```python
# cookbook_ops.py
def judge(client, version, log=print):
    """07's judge over this version's saved answers; "points made" goes onto each answer's trace as a score."""
    verdicts, finished = J.judge_all(saved(version), _path("verdicts", version), log=log)
    questions = {q["question"]: q for q in AGENT_QUESTIONS}
    for question, verdict in verdicts.items():
        trace = saved(version)[question]["trace"]
        if "points made" in verdict and not verdict.get("scored"):
            client.create_score(trace_id=trace, name="points made", data_type="NUMERIC",
                                value=J.scores(verdict, questions[question])["points made"],
                                comment=f"{len(verdict['points made'])} of {len(questions[question]['points'])} "
                                        f"reference points, by {verdict['judge']}")
            verdict["scored"] = True
    _path("verdicts", version).write_text(json.dumps(verdicts, indent=1, ensure_ascii=False), encoding="utf-8")
    client.flush()
    return verdicts, finished
```

---
# Part 2 · Run and show

In [1]:
import asyncio
import json
import warnings
from datetime import datetime, timedelta, timezone

warnings.filterwarnings("ignore")

import cookbook_ops as O        # first: it puts 05 and 07 on the path
import ask_index as A
import claim_judge as J

client = O.setup()


def show(value):
    print(json.dumps(value, indent=1, ensure_ascii=False, default=str))


def steps(trace=None, environment="default", since=timedelta(days=7)):
    # every observation of a trace (or of an environment), from Langfuse's v2 observations API
    now = datetime.now(timezone.utc)
    return client.api.observations.get_many(trace_id=trace, environment=environment, from_start_time=now - since,
                                            to_start_time=now, limit=200, fields="core,basic,usage,prompt,model").data

## 1 · Offline first, then a stand-in run

The offline tests (`test_ops.py`) run the whole thing against a scripted Groq and a fake Langfuse client. Then one question went through to the real Langfuse project with the scripted Groq, in an environment of its own (`stand-in`), so it never mixes with the real runs. This is what Langfuse recorded for its model calls: no Groq token was spent.

In [2]:
standin = sorted(steps(environment="stand-in"), key=lambda o: o.start_time)
by_id = {o.id: o for o in standin}
for o in standin:
    if o.type == "GENERATION":
        parent = by_id.get(o.parent_observation_id)
        print(f"{o.name:44} inside {parent.name if parent else '-':40} tokens={o.usage_details or '-'}"
              f"  cost={o.total_cost}  prompt={o.prompt_name} v{o.prompt_version}")

MeteredOpenAILike._prepare_chat_with_tools   inside BaseWorkflowAgent.run_agent_step         tokens=-  cost=None  prompt=cookbook-agent v2
MeteredOpenAILike.achat                      inside BaseWorkflowAgent.run_agent_step         tokens=-  cost=None  prompt=cookbook-agent v2
MeteredOpenAILike.achat                      inside MeteredOpenAILike.achat                  tokens=-  cost=None  prompt=cookbook-agent v2
MeteredOpenAILike.achat                      inside MeteredOpenAILike.achat                  tokens={'input': 900, 'output': 60, 'total': 960}  cost=0.000171  prompt=cookbook-agent v2
MeteredOpenAILike._prepare_chat_with_tools   inside BaseWorkflowAgent.run_agent_step         tokens=-  cost=None  prompt=cookbook-agent v2
MeteredOpenAILike.achat                      inside BaseWorkflowAgent.run_agent_step         tokens=-  cost=None  prompt=cookbook-agent v2
MeteredOpenAILike.achat                      inside MeteredOpenAILike.achat                  tokens=-  cost=None  prompt=

Two things a reader of these numbers has to know:

- **One model call is four "generations".** LlamaIndex's instrumentation records a call at each layer it passes through: preparing the tool schema, the agent's call, and the model's call inside it. 05's metered model adds one more, because its `achat` wraps LlamaIndex's. Only the innermost carries the tokens and the cost, so sums of tokens and cost are right, and a **count** of generations is four times the number of calls.
- **Every one of them is linked to the prompt version**, through `propagate_attributes`.

The stand-in's 900 tokens in and 60 out, at Groq's prices, are $0.000171 a call, which is what Langfuse shows.

## 2 · The price and the prompts, in Langfuse

In [2]:
price = O.ensure_price(client)
print(f"{price.model_name}: ${price.input_price * 1e6:.2f} per million tokens in, ${price.output_price * 1e6:.2f} out"
      f"  (matches {price.match_pattern})")
for version, prompt in O.ensure_prompts(client).items():
    print(f"\n{O.PROMPT} v{prompt.version}, labels {prompt.labels}:\n{prompt.prompt}")

openai/gpt-oss-120b: $0.15 per million tokens in, $0.60 out  (matches (?i)^(openai/)?gpt-oss-120b$)



cookbook-agent v1, labels ['v1']:
You answer questions about agentic-forge-cookbook, a repo that teaches agent engineering in chapters
(01_foundations to 08_frameworks). Use search_cookbook to find passages -- more than once if the question has several
parts -- and open_file to read on when a passage stops short. Answer only from what you found, and cite the files you
used in square brackets, like [04_memory/recall.py]. If the cookbook does not cover the question, say so plainly.

cookbook-agent v2, labels ['v2', 'latest']:
You answer questions about agentic-forge-cookbook, a repo that teaches agent engineering in chapters
(01_foundations to 08_frameworks). Use search_cookbook to find passages -- more than once if the question has several
parts -- and open_file to read on when a passage stops short. Answer only from what you found, and cite the files you
used in square brackets, like [04_memory/recall.py]. If the cookbook does not cover the question, say so plainly.

When the question

## 3 · Version 1, live

The three agent questions, with 05's own prompt.

In [2]:
index = A.open_index()
await O.run_version(client, index, 1)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


    v1: How is chapter 6's shared board different from chapter 4's m -- 44,323 tokens


    v1: In chapter 8's LangGraph project, compare how the LangGraph  -- 29,574 tokens


    v1: How does chapter 2's runtime stop an agent that keeps callin -- 14,497 tokens


## 4 · Version 2, live

The same three questions, with the revised prompt.

In [3]:
await O.run_version(client, index, 2)

    v2: How is chapter 6's shared board different from chapter 4's m -- 31,947 tokens


    v2: In chapter 8's LangGraph project, compare how the LangGraph  -- 36,091 tokens


    v2: How does chapter 2's runtime stop an agent that keeps callin -- 8,318 tokens


**Three of the six answers never came.** What each run ended with, from the saved results:

In [2]:
for version in (1, 2):
    O.set_aside_timeouts(version)
    for q in O.AGENT_QUESTIONS:
        a = O.saved(version).get(q["question"]) or O.timed_out(version).get(q["question"])
        print(f"v{version}  {'answered' if a['answer'] else 'NO ANSWER':9}  tool calls={len(a['calls'])}"
              f"  tokens={a['usage']['total']:>6,}  seconds={a['seconds']:>6}  {q['question'][:50]}")
        if a["stopped"]:
            print("      stopped:", a["stopped"][:110])

v1  answered   tool calls=8  tokens=44,323  seconds= 270.0  How is chapter 6's shared board different from cha
v1  NO ANSWER  tool calls=8  tokens=29,574  seconds= 300.8  In chapter 8's LangGraph project, compare how the 
      stopped: WorkflowTimeoutError: Operation timed out after 300.0 seconds. Currently active steps: run_agent_step
v1  answered   tool calls=4  tokens=14,497  seconds=  81.2  How does chapter 2's runtime stop an agent that ke
v2  answered   tool calls=7  tokens=31,947  seconds= 218.0  How is chapter 6's shared board different from cha
v2  NO ANSWER  tool calls=8  tokens=36,091  seconds= 301.6  In chapter 8's LangGraph project, compare how the 
      stopped: WorkflowTimeoutError: Operation timed out after 300.0 seconds. Currently active steps: parse_agent_output
v2  NO ANSWER  tool calls=4  tokens= 8,318  seconds=1025.0  How does chapter 2's runtime stop an agent that ke
      stopped: WorkflowTimeoutError: Operation timed out after 300.0 seconds. Currently active s

**The time limit, not the size limit, now ends long runs.** LlamaIndex gives an agent run 300 seconds (05 set `timeout=300`), and the LangGraph question ran out of them with both versions, after eight tool calls; so did the third question with version 2. The calls themselves are quick. The waiting is not: fix 3 in 07 keeps every request near 6,000 tokens, Groq's free tier allows 8,000 tokens a minute, and so each call after the first waits about 45 seconds for the minute to free up (the OpenAI client waits out each 429 by itself, silently). A nine-call question is six or seven minutes, nearly all of it waiting. In 07's round 3 the same question had finished just inside the limit.

One run took 1,025 seconds to stop: a single Groq request hung, and a request is allowed 120 seconds and retried up to 8 times. LlamaIndex's time limit cannot interrupt a call that is still waiting on the network.

The traces of all six runs are in Langfuse, failures included. That is what monitoring sees: what was spent, not only what succeeded. The fix is in 05: a run gets 900 seconds.

```python
# ask_index.py
# How long LlamaIndex lets one agent run take (its workflow `timeout`). It was 300 seconds. Once fit() keeps every
# request near 6,000 tokens, Groq's free-tier limit of 8,000 tokens a minute spaces the calls about 45 seconds
# apart -- the client waits out each 429 -- so a nine- or ten-call question takes six to seven minutes, nearly all of
# it waiting. 08_langfuse's runs lost the LangGraph question to the 300-second limit with both prompt versions.
AGENT_TIMEOUT = 900
```

The three cut-off answers again, with the longer limit. The answers that came are kept, and are not asked again.

In [2]:
index = A.open_index()
for version in (1, 2):
    await O.run_version(client, index, version)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


    v1: In chapter 8's LangGraph project, compare how the LangGraph  -- 43,716 tokens


    v2: In chapter 8's LangGraph project, compare how the LangGraph  -- 36,400 tokens


    v2: How does chapter 2's runtime stop an agent that keeps callin -- 12,985 tokens


**Two of the three came; one failed in a new way.** No run timed out this time. Version 2's LangGraph question went through its eight tool calls, and then failed at the step-limit answer: the request with no tools offered. gpt-oss called a tool anyway, Groq refused it (`Tool choice is none, but model called a tool`), and the two retries got the same. After eight tool calls the conversation is all tool calls, and the model keeps making them. Asking the identical request again cannot change that; saying so in it can. So 05's step-limit answer (fix 4 in 07) now has one more step: if that request is refused for a tool call, it is asked once more with a plain instruction added.

In [2]:
for version in (1, 2):
    for q, a in O.set_aside_unanswered(version, "asked again after NO_TOOLS_NOW was added to the step-limit answer").items():
        print(f"v{version} set aside: {q[:60]}\n      stopped: {a['stopped'][:120]}")

v2 set aside: In chapter 8's LangGraph project, compare how the LangGraph 
      stopped: BadRequestError: Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 'invali


```python
# ask_index.py
NO_TOOLS_NOW = ("No tools are available now. Answer the question in plain text from what you have found, and cite "
                "the files it came from.")
```

```python
# ask_index.py
def _finishing():
    """LlamaIndex's `FunctionAgent`, with its step-limit answer given what the run found.

    `early_stopping_method="generate"` makes one last request when `max_iterations` is reached, with no tools, for an
    answer. LlamaIndex builds it from the agent's memory -- and during a run, the tool calls and results are not in
    memory but in the step "scratchpad", moved into memory only when the run finishes normally (`finalize`). So the
    last request carried the system prompt, the question and the stop notice, and nothing the eight tool calls had
    found: an answer written blind (test_ask.test_no_agent_request_outgrows_its_limit, which first showed it).

    So what the run found goes into memory first -- as one message of plain text (`_found`), not as the run's tool
    calls and results. The first version of this fix moved the tool messages in as they were, and on 08_langfuse's
    LangGraph question (prompt v2) gpt-oss answered the no-tools request with yet another tool call, which Groq
    refuses, on every try -- including two after NO_TOOLS_NOW was added: a request that is all tool calls invites
    one more. With the results as text there is no call in the request to continue. If the model calls a tool
    anyway, it is asked once more with NO_TOOLS_NOW said plainly.
    """
    global _FINISHING
    if _FINISHING is None:
        from llama_index.core.agent.workflow import FunctionAgent

        class FinishingAgent(FunctionAgent):
            async def _generate_early_stopping_response(self, ctx, max_iterations):
                import openai
                from llama_index.core.base.llms.types import ChatMessage
                scratchpad = await ctx.store.get(self.scratchpad_key, default=[])
                memory = await ctx.store.get("memory")
                await memory.aput(ChatMessage(role="user", content=_found(scratchpad)))
                await ctx.store.set(self.scratchpad_key, [])
                try:
                    return await super()._generate_early_stopping_response(ctx, max_iterations)
                except openai.BadRequestError as error:
                    if not _unparsed(error) and "Tool choice is none" not in str(error):
                        raise
                    await memory.aput(ChatMessage(role="user", content=NO_TOOLS_NOW))
                    return await super()._generate_early_stopping_response(ctx, max_iterations)

        _FINISHING = FinishingAgent
    return _FINISHING
```

The one failed answer, asked again:

In [2]:
index = A.open_index()
await O.run_version(client, index, 2)      # only what is not saved: the one set aside

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


    v2: In chapter 8's LangGraph project, compare how the LangGraph  -- 35,230 tokens


**The instruction did not help.** The step-limit answer was refused for a tool call twice as asked, and twice more with "No tools are available now" added. The request itself was the problem: it carried the run's eight tool calls and their results, and the model continued the pattern. So the step-limit answer now gets what the run found **as plain text**, in one message, with the newest results whole and older ones cut to their labels, and no tool call anywhere in the request. The instruction stays, as a second try.

In [2]:
for q, a in O.set_aside_unanswered(2, "asked again after the step-limit answer got the findings as plain text").items():
    print(f"v2 set aside: {q[:60]}\n      stopped: {a['stopped'][:120]}")

v2 set aside: In chapter 8's LangGraph project, compare how the LangGraph 
      stopped: BadRequestError: Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 'invali


```python
# ask_index.py
def _finishing():
    """LlamaIndex's `FunctionAgent`, with its step-limit answer given what the run found.

    `early_stopping_method="generate"` makes one last request when `max_iterations` is reached, with no tools, for an
    answer. LlamaIndex builds it from the agent's memory -- and during a run, the tool calls and results are not in
    memory but in the step "scratchpad", moved into memory only when the run finishes normally (`finalize`). So the
    last request carried the system prompt, the question and the stop notice, and nothing the eight tool calls had
    found: an answer written blind (test_ask.test_no_agent_request_outgrows_its_limit, which first showed it).

    So what the run found goes into memory first -- as one message of plain text (`_found`), not as the run's tool
    calls and results. The first version of this fix moved the tool messages in as they were, and on 08_langfuse's
    LangGraph question (prompt v2) gpt-oss answered the no-tools request with yet another tool call, which Groq
    refuses, on every try -- including two after NO_TOOLS_NOW was added: a request that is all tool calls invites
    one more. With the results as text there is no call in the request to continue. If the model calls a tool
    anyway, it is asked once more with NO_TOOLS_NOW said plainly.
    """
    global _FINISHING
    if _FINISHING is None:
        from llama_index.core.agent.workflow import FunctionAgent

        class FinishingAgent(FunctionAgent):
            async def _generate_early_stopping_response(self, ctx, max_iterations):
                import openai
                from llama_index.core.base.llms.types import ChatMessage
                scratchpad = await ctx.store.get(self.scratchpad_key, default=[])
                memory = await ctx.store.get("memory")
                await memory.aput(ChatMessage(role="user", content=_found(scratchpad)))
                await ctx.store.set(self.scratchpad_key, [])
                try:
                    return await super()._generate_early_stopping_response(ctx, max_iterations)
                except openai.BadRequestError as error:
                    if not _unparsed(error) and "Tool choice is none" not in str(error):
                        raise
                    await memory.aput(ChatMessage(role="user", content=NO_TOOLS_NOW))
                    return await super()._generate_early_stopping_response(ctx, max_iterations)

        _FINISHING = FinishingAgent
    return _FINISHING

def _found(scratchpad, limit=FOUND_TOKENS):
    """The run's tool results as one plain-text message for the step-limit answer: the newest whole, older ones cut
    to their labels (file, section, lines) once the whole would pass `limit` tokens -- fit()'s rule, in text."""
    from llama_index.core.base.llms.types import ChatMessage
    results = [m.content or "" for m in scratchpad if m.role.value == "tool"]
    kept, used = [], 0
    for text in reversed(results):
        size = _tokens(ChatMessage(role="user", content=text))
        if used + size > limit:
            text = _labels(text)
            size = _tokens(ChatMessage(role="user", content=text))
        kept.append(text)
        used += size
    return ("You can make no more tool calls. What your tool calls found, newest last (older results cut to "
            "their labels):\n\n" + "\n\n---\n\n".join(reversed(kept)))
```

The one failed answer, asked a third time:

In [2]:
index = A.open_index()
await O.run_version(client, index, 2)      # the third try at the one answer that has not come

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


    v2: In chapter 8's LangGraph project, compare how the LangGraph  -- 43,257 tokens


## 5 · One answer's trace

The LangGraph question with version 2, read back from Langfuse: every step, nested as it ran.

In [2]:
question = O.AGENT_QUESTIONS[1]["question"]
trace = O.saved(2)[question]["trace"]
tree = steps(trace)
children = {}
for o in tree:
    children.setdefault(o.parent_observation_id, []).append(o)


from collections import Counter

print(f"{len(tree)} steps in all:", dict(Counter(o.type for o in tree)))
calls = [o for o in tree if o.usage_details]
print(f"{len(calls)} model calls carry tokens: {sum(o.usage_details['input'] for o in calls):,} in,"
      f" {sum(o.usage_details['output'] for o in calls):,} out, ${sum(o.total_cost or 0 for o in calls):.4f}\n")

shown = [0]


def walk(parent, depth):
    # the run's first two passes through the agent's loop, every step nested as it ran
    for o in sorted(children.get(parent, []), key=lambda o: o.start_time):
        if depth == 2 and o.name.endswith("setup_agent"):
            shown[0] += 1
            if shown[0] > 2:
                return
        cost = f"  ${o.total_cost:.5f}" if o.total_cost else ""
        tokens = f"  {o.usage_details['input']:,} in / {o.usage_details['output']:,} out" if o.usage_details else ""
        print(f"{'  ' * depth}{o.type[:4]} {o.name}{tokens}{cost}")
        walk(o.id, depth + 1)


walk(None, 0)
print("   ... and the same for each later pass, then the answer at the step limit")

165 steps in all: {'GENERATION': 39, 'CHAIN': 45, 'TOOL': 8, 'RETRIEVER': 60, 'EMBEDDING': 12, 'SPAN': 1}
10 model calls carry tokens: 37,075 in, 1,750 out, $0.0066

SPAN cookbook-agent
  CHAI FinishingAgent.run
    CHAI BaseWorkflowAgent.init_run
    CHAI BaseWorkflowAgent.setup_agent
    CHAI BaseWorkflowAgent.run_agent_step
      GENE MeteredOpenAILike._prepare_chat_with_tools
      GENE MeteredOpenAILike.achat
        GENE MeteredOpenAILike.achat
          GENE MeteredOpenAILike.achat  494 in / 38 out  $0.00010
    CHAI BaseWorkflowAgent.parse_agent_output
    CHAI BaseWorkflowAgent.call_tool
      TOOL FunctionTool.acall
        RETR CookbookRetriever.retrieve
          RETR CookbookRetriever._retrieve
            RETR VectorIndexRetriever.retrieve
              RETR VectorIndexRetriever._retrieve
                EMBE GoogleGenAIEmbedding.get_query_embedding
                  EMBE GoogleGenAIEmbedding._get_query_embedding
            RETR QueryFusionRetriever.retrieve
            

## 6 · Monitoring: the two versions compared

Cost and tokens per version, from the metrics API: the sums over the model calls that carry usage.

In [3]:
for version in (1, 2):
    rows = O.metrics(client, ["promptVersion"],
                     [("totalCost", "sum"), ("inputTokens", "sum"), ("outputTokens", "sum"), ("latency", "p50")],
                     filters=[O.tagged(version), {"column": "type", "operator": "=", "value": "GENERATION", "type": "string"}])
    show(rows)

[
 {
  "promptVersion": 1,
  "sum_totalCost": 0.02067029999,
  "sum_inputTokens": 123242,
  "sum_outputTokens": 8868,
  "p50_latency": 7785
 }
]


[
 {
  "promptVersion": 2,
  "sum_totalCost": 0.034049699991,
  "sum_inputTokens": 220610,
  "sum_outputTokens": 12798,
  "p50_latency": 11894
 }
]


## 7 · The answers, scored

07's judge, points made, on each answer's trace. Saved one by one; if Gemini's daily limit is reached, run again after the reset.

In [2]:
for version in (1, 2):
    O.judge(client, version)
for q in O.AGENT_QUESTIONS:
    print(q["question"][:90])
    for version in (1, 2):
        a, v = O.saved(version).get(q["question"]), O.saved(version, "verdicts").get(q["question"])
        made = f"points made {J.scores(v, q)['points made']:.0%}" if v and "points made" in v else "not judged yet"
        print(f"   v{version}  {made:24} tokens={a['usage']['total']:>6,}  tool calls={len(a['calls'])}"
              + (f"  stopped: {a['stopped'][:50]}" if a["stopped"] else ""))

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


    judged 3 of 3: In chapter 8's LangGraph project, compare how the LangGraph 


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


    judged 1 of 3: How is chapter 6's shared board different from chapter 4's m


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


    judged 2 of 3: How does chapter 2's runtime stop an agent that keeps callin


    judged 3 of 3: In chapter 8's LangGraph project, compare how the LangGraph 


How is chapter 6's shared board different from chapter 4's memory stores? When would you u
   v1  points made 100%         tokens=44,323  tool calls=8
   v2  points made 100%         tokens=31,947  tool calls=7
In chapter 8's LangGraph project, compare how the LangGraph desk and the from-scratch case
   v1  points made 71%          tokens=43,716  tool calls=8
   v2  points made 86%          tokens=43,257  tool calls=8
How does chapter 2's runtime stop an agent that keeps calling the same tool, and how does 
   v1  points made 71%          tokens=14,497  tool calls=4
   v2  points made 71%          tokens=12,985  tool calls=4


## Findings

**Langfuse gave the four things 07 did not show, with no code of ours on the spans.**

- **Tracing.** Langfuse is an OpenTelemetry tracer, so LlamaIndex's own instrumentation shows every step, nested where it ran: 165 steps for one answer. In 07 that same route lost every LlamaIndex step inside a LangSmith experiment.
- **Cost.** One entry in the price table, Groq's $0.15 and $0.60 per million tokens, and every model call shows dollars. All of 08's runs, retries and failures included, came to about $0.055.
- **Prompt versions.** The agent's instructions live in Langfuse as numbered versions, and `propagate_attributes(prompt=...)` links every model call to the version it ran with.
- **Monitoring.** One query shape (measure, aggregation, group, filters) answers cost, tokens and time by prompt version, from code as well as from the dashboards.

**Know this before reading the numbers:**

- One model call is recorded as four "generations": LlamaIndex's layers, and 05's metered model wrapping its call. Sums of tokens and cost are right; a count of generations is four times the number of calls.
- New organizations cannot use Langfuse's older trace API; reading back goes through the v2 observations API. The older API is being retired on 16 November 2026.

**Prompt v2 against v1,** one run each:

| | points made, v1 | points made, v2 |
|---|---|---|
| Board vs stores | 100% | 100% |
| LangGraph waiting | 71% | 86% |
| Runtime vs LangGraph | 71% | 71% |

v2 made one more point, on the LangGraph question: the one its revision was written from (cover each side, and what each gives up). Elsewhere it changed nothing, and it cost more: $0.034 against $0.021, mostly because its LangGraph answer took three tries. Since v2 was written from misses on these very questions, the result shows the revision does what it was written for, not that it would help on others. And three questions, one run each, is what the free tier allows, not what the comparison needs: one point on one question is one sample.

**Running the agent for real found two more limits, each fixed in 05:**

- **Time.** LlamaIndex gave a run 300 seconds. With every request near 6,000 tokens, Groq's 8,000-tokens-a-minute limit spaces the calls about 45 seconds apart, so a nine-call question takes six or seven minutes, nearly all of it waiting. Three of the six first answers ran out of time. One hung request took 1,025 seconds to stop. A run now gets 900 seconds.
- **The step-limit answer.** Asked for an answer with no tools, after eight tool calls given back as tool messages, gpt-oss called a tool again, which Groq refuses, on every try, including twice after being told plainly that no tools were left. Given what the run found as plain text, it answered at once.

Monitoring is what showed both. Each of the six questions passes on a good run. What the traces and the cost table showed is how often a run is not a good one, and what it costs when it isn't.